# AI Job Recommendation & Skill Gap Analyzer
## Exploratory Analysis & ML Walkthrough

> **Purpose:** This notebook demonstrates every step of the ML pipeline in a transparent, explainable way for academic viva and presentation.

**Pipeline:**
```
Dataset → Cleaning → Preprocessing → Skill Extraction → TF-IDF → Cosine Similarity → Recommendations → Skill Gap → Priority
```

## 1. Import Libraries

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.facecolor'] = '#1a1a2e'
matplotlib.rcParams['figure.facecolor'] = '#16213e'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.preprocessing import load_and_preprocess, clean_text
from src.skill_extraction import extract_skills, SKILL_VOCABULARY
from src.recommender import JobRecommender, precision_at_k
from src.skill_gap import analyse_skill_gap, priority_skills

print('All libraries imported successfully!')

## 2. Load & Inspect Dataset

In [ ]:
DATA_PATH = os.path.join(os.getcwd(), '..', 'data', 'jobs.csv')

# Load raw data to inspect
raw_df = pd.read_csv(DATA_PATH, low_memory=False, nrows=5000)
print(f'Raw Shape: {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')
raw_df.head(3)

In [ ]:
# Check missing values
print('Missing values per column:')
raw_df.isnull().sum().sort_values(ascending=False)

## 3. Data Preprocessing

In [ ]:
# Run the full preprocessing pipeline
df = load_and_preprocess(DATA_PATH, sample_size=2000)
print(f'\nCleaned shape: {df.shape}')
df[['title', 'company', 'location', 'skills', 'category']].head(5)

In [ ]:
# Show text cleaning example
sample_raw = '  Python Developer @ Google (NY)! Req: FastAPI, Docker, REST API !!!  '
sample_clean = clean_text(sample_raw)
print(f'Raw  : {sample_raw}')
print(f'Clean: {sample_clean}')

## 4. Exploratory Data Analysis

In [ ]:
# Top job categories
top_cats = df['category'].value_counts().head(10)

colors = plt.cm.Purples(np.linspace(0.4, 0.9, len(top_cats)))
fig, ax = plt.subplots()
top_cats[::-1].plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_xlabel('Number of Jobs')
ax.set_title('Top 10 Job Categories in Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Top skills across all jobs
from collections import Counter

all_skills = []
for text in df['combined_text'].dropna():
    all_skills.extend(extract_skills(text))

skill_counts = Counter(all_skills).most_common(15)
skills_df = pd.DataFrame(skill_counts, columns=['skill', 'count'])

colors2 = plt.cm.plasma(np.linspace(0.3, 0.9, len(skills_df)))
fig, ax = plt.subplots()
ax.barh(skills_df['skill'][::-1], skills_df['count'][::-1], color=colors2[::-1])
ax.set_xlabel('Frequency in Dataset')
ax.set_title('Top 15 Skills Demanded Across All Jobs')
plt.tight_layout()
plt.show()

## 5. Skill Extraction Demo

In [ ]:
sample_text = """
We are looking for a Data Engineer with strong skills in Python, SQL, and Apache Spark.
Experience with AWS, Docker, and Kubernetes is required. 
Knowledge of Airflow and dbt is a plus.
"""

extracted = extract_skills(sample_text)
print('Extracted skills from job description:')
for s in extracted:
    print(f'  ✓ {s.title()}')

## 6. TF-IDF Vectorisation — Step by Step

In [ ]:
# Demonstrate TF-IDF with a tiny example
documents = [
    'python sql machine learning data analysis',
    'java spring docker kubernetes microservices',
    'python django flask rest api postgresql',
    'machine learning tensorflow pytorch deep learning',
]

demo_vectorizer = TfidfVectorizer()
demo_matrix = demo_vectorizer.fit_transform(documents)

demo_df = pd.DataFrame(
    demo_matrix.toarray().round(3),
    columns=demo_vectorizer.get_feature_names_out(),
    index=[f'Job {i+1}' for i in range(len(documents))]
)

print('TF-IDF Matrix (mini demo):')
print('Each cell = TF × IDF score for that term in that document')
demo_df

## 7. Cosine Similarity — Step by Step

In [ ]:
# User profile
user_profile = 'python sql data analysis machine learning'
user_vec = demo_vectorizer.transform([user_profile])

# Compute cosine similarity
sims = cosine_similarity(user_vec, demo_matrix).flatten()

print('Cosine Similarity between user profile and each job:')
for i, sim in enumerate(sims):
    print(f'  Job {i+1}: {sim:.4f}  ({sim*100:.1f}% match)')
    print(f'    → {documents[i]}')

print(f'\n→ Best match: Job {np.argmax(sims)+1} ({sims.max()*100:.1f}%)')

In [ ]:
# Visualise cosine similarity
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'Job {i+1}' for i in range(len(sims))], sims * 100,
              color=['#7f00ff' if i == np.argmax(sims) else '#4a4a8a' for i in range(len(sims))])
ax.set_ylabel('Cosine Similarity (%)')
ax.set_title('Cosine Similarity: User Profile vs Jobs')
ax.set_ylim(0, 100)
for bar, val in zip(bars, sims * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontsize=11, color='white')
plt.tight_layout()
plt.show()

## 8. Full Recommender — Real Dataset

In [ ]:
# Fit the recommender on the real dataset
rec = JobRecommender()
rec.fit(df)
print('Recommender fitted!')

In [ ]:
# Test with a sample user profile
user_skills = ['python', 'sql', 'pandas', 'machine learning', 'git']
user_role   = 'data analyst'
user_text   = ' '.join(user_skills) + ' ' + user_role

results = rec.recommend(user_text, top_k=10)
results[['title', 'company', 'location', 'match_score', 'matching_skills']]

In [ ]:
# Evaluation
p_at_k = precision_at_k(results, threshold=15.0)
print(f'Precision@K (threshold=15%): {p_at_k:.2%}')
print(f'Match score range: {results["match_score"].min():.1f}% – {results["match_score"].max():.1f}%')
print(f'Average match score: {results["match_score"].mean():.1f}%')

## 9. Skill Gap Analysis Demo

In [ ]:
# Analyse gap for the top job
top_job = results.iloc[0]
gap = analyse_skill_gap(user_skills, top_job['job_skills'])

print(f'\nJob: {top_job["title"].title()}')
print(f'Match %: {gap["match_pct"]}%\n')

print('✅ Matching skills:')
for s in gap['matching']:
    print(f'   ✓ {s.title()}')

print('\n⚠️ Missing skills:')
for s in gap['missing']:
    print(f'   ✗ {s.title()}')

## 10. Priority Skill Recommendations

In [ ]:
priorities = priority_skills(results, user_skills, top_n=10)

print('Priority skills to learn (based on frequency across top jobs):')
print('─' * 50)
for item in priorities:
    bar = '█' * item['frequency']
    print(f"Priority {item['priority']:>2} │ {item['skill'].title():<25} │ {bar} ({item['frequency']} jobs)")

In [ ]:
# Visualise priorities
prio_df = pd.DataFrame(priorities)
colors3 = plt.cm.Reds(np.linspace(0.4, 0.9, len(prio_df)))

fig, ax = plt.subplots()
ax.barh(prio_df['skill'].str.title()[::-1], prio_df['frequency'][::-1], color=colors3)
ax.set_xlabel('Appears in N recommended jobs')
ax.set_title('Priority Missing Skills to Learn')
plt.tight_layout()
plt.show()

## Summary

| Step | Method | Library |
|------|--------|---------|
| Data Loading | `pd.read_csv()` | Pandas |
| Missing Values | `fillna()` | Pandas |
| Deduplication | `drop_duplicates()` | Pandas |
| Text Cleaning | regex `re.sub()` | Python built-in |
| Skill Extraction | Rule-based vocabulary matching | Python built-in |
| Vectorisation | TF-IDF | Scikit-learn |
| Similarity | Cosine Similarity | Scikit-learn |
| Skill Gap | Set operations (`∩`, `-`) | Python built-in |
| Priority | `Counter.most_common()` | Python built-in |
| Evaluation | Precision@K | Manual calculation |

**Note:** Every step uses well-understood, explainable methods. There is no hidden AI or black-box model.